# Contribution 1: GMM soft intent clustering (a methodological alternative
to hard K-means)

**Course:** CCAI 422 Recommender Systems — University of Jeddah

**Goal:** Replace ICSRec's hard K-means intent clustering with a **Gaussian Mixture Model (GMM)** that performs *soft* clustering.

### Motivation

ICSRec assumes each user behaviour subsequence belongs to exactly **one** latent intent (hard assignment via K-means). But real grocery shopping is **multi-intent**: a single basket mixes breakfast items, snacks, cleaning supplies, etc. A Gaussian Mixture Model captures this by assigning each subsequence a *probability distribution* over intents (soft assignment) and modelling each intent as a Gaussian with its own covariance, which is a more expressive representation of overlapping shopping intentions.

### What we change

We add a `GMM` class with the **exact same interface** as the original `KMeans` class (`train(x)` and `query(x)`), then point the trainer at it. Everything else in ICSRec stays identical, so the comparison is clean: same data, same model, same hyperparameters — only the intent-clustering mechanism differs.

### Why we use `intent_num=128` (not the best `K=512`)

Our hyperparameter sweep identified **`K=512` as the best K-means configuration** (HIT@20 = 0.0842, NDCG@20 = 0.0357). We initially planned to compare GMM against K-means at this best setting, but ran into two practical issues:

1. **CPU bottleneck.** ICSRec's K-means runs on GPU via `faiss`, which we patched to `faiss-cpu` for Colab compatibility. The GMM is implemented through `sklearn.mixture.GaussianMixture`, which is **CPU-only**. Fitting a 512-component GMM at the start of every training epoch (over ~12,700 subsequence embeddings in 64-dim space) was prohibitively slow — each clustering step took over a minute, multiplying training time several-fold.

2. **What the comparison actually tests.** The purpose of this experiment is to isolate the effect of *soft vs hard* intent assignment, not the effect of cluster *capacity*. Running both methods at `K=128` is in fact the **more informative comparison**: it controls for capacity and isolates the assignment mechanism. If GMM wins here, the win is attributable to soft assignment specifically; if it ties or loses, hard K-means is doing fine on the same capacity budget.

So we compare **GMM vs K-means at `K=128`** — the same value we used in one of our tuning runs, giving us a clean head-to-head. The headline tuning result (`K=512` is best for K-means) is reported separately in the hyperparameter sweep notebook and the report.

## Step 1: Environment setup (same as before)

In [1]:
import torch, os, shutil, subprocess
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU: Runtime -> Change runtime type'

# Mount Drive
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/ICSRec_project'

# Install deps
for pkg in ['faiss-cpu', 'gensim', 'scikit-learn']:
    subprocess.run(['pip', 'install', '-q', pkg])
print('Packages installed (faiss-cpu, gensim, scikit-learn)')

# Clone repo if missing
if not os.path.exists('/content/ICSRec/src/main.py'):
    subprocess.run(['git', 'clone', 'https://github.com/QinHsiu/ICSRec.git', '/content/ICSRec'])
    print('Cloned ICSRec')

# Restore data files from Drive
for fname in ['Grocery_and_Gourmet_Food.txt', 'Grocery_and_Gourmet_Food_1.txt']:
    src = os.path.join(DRIVE_DIR, fname)
    dst = f'/content/ICSRec/data/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f'Restored {fname}')
print('Setup complete.')

PyTorch: 2.11.0+cu128 | CUDA: True
Packages installed (faiss-cpu, gensim, scikit-learn)
Setup complete.


## Step 2: Inject the GMM class into models.py

We append a `GMM` class to `models.py`. It uses scikit-learn's `GaussianMixture` under the hood but exposes the **same `train(x)` / `query(x)` interface** as the original `KMeans`, so it drops straight into ICSRec.

Key design points:
- `train(x)`: fits a GMM with `num_cluster` components on the user embeddings. We use a `diag` covariance (one variance per dimension) which is fast and stable for 512 components in 64-dim space.
- `query(x)`: for each input embedding, computes the **most likely** Gaussian component (the soft assignment's argmax) and returns that component's mean as the intent prototype — keeping the return signature identical to K-means `(seq2cluster, centroids[seq2cluster])`.
- The means are L2-normalised, matching what the original K-means did.

In [2]:
gmm_class_code = '''

# ============================================================
# METHODOLOGY IMPROVEMENT (Option C): GMM soft-clustering
# Drop-in replacement for KMeans with identical interface.
# Added for CCAI422 course project.
# ============================================================
from sklearn.mixture import GaussianMixture
import numpy as _np

class GMM(object):
    def __init__(self, num_cluster, seed, hidden_size, gpu_id=0, device="cpu"):
        self.seed = seed
        self.num_cluster = num_cluster
        self.hidden_size = hidden_size
        self.device = device
        self.gmm = None
        self.centroids = []  # will hold the (normalised) Gaussian means
        print(" Using GMM soft clustering with", num_cluster, "components")

    def train(self, x):
        # x: numpy array [N, hidden_size] of user-intent embeddings
        if x.shape[0] <= self.num_cluster:
            # Not enough points to fit all components; reduce components defensively
            n_comp = max(2, x.shape[0] // 2)
        else:
            n_comp = self.num_cluster
        self.gmm = GaussianMixture(
            n_components=n_comp,
            covariance_type='diag',   # fast & stable for high #components
            max_iter=20,              # match K-means niter=20 for fair compute
            n_init=1,
            random_state=self.seed,
            reg_covar=1e-4,           # numerical stability
        )
        self.gmm.fit(x)
        means = torch.Tensor(self.gmm.means_).to(self.device)
        # L2-normalise the means, exactly like the original KMeans did to its centroids
        self.centroids = nn.functional.normalize(means, p=2, dim=1)

    def query(self, x):
        # x: numpy array [B, hidden_size]
        # predict() returns the most-likely component (argmax of soft posterior)
        labels = self.gmm.predict(x)
        seq2cluster = torch.LongTensor(labels).to(self.device)
        return seq2cluster, self.centroids[seq2cluster]
'''

# append the GMM class to models.py
models_path = '/content/ICSRec/src/models.py'
with open(models_path) as f:
    content = f.read()

if 'class GMM(object):' not in content:
    with open(models_path, 'a') as f:
        f.write(gmm_class_code)
    print('GMM class appended to models.py')
else:
    print('GMM class already present in models.py')

GMM class already present in models.py


## Step 3: Point the trainer at GMM instead of KMeans

In `trainers.py`, the trainer imports `KMeans` and instantiates it. We add a command-line switch `--cluster_method` so we can choose `kmeans` (original) or `gmm` (our improvement) without breaking the original code. This is cleaner than hard-editing and lets us run both for comparison.

In [3]:
trainers_path = '/content/ICSRec/src/trainers.py'
with open(trainers_path) as f:
    tcode = f.read()

# 3a. Update the import to also bring in GMM
if 'from models import KMeans' in tcode and 'GMM' not in tcode.split('from models import')[1].split('\n')[0]:
    tcode = tcode.replace('from models import KMeans', 'from models import KMeans, GMM')
    print('Updated import to include GMM')

# 3b. Replace the cluster instantiation with a switch on args.cluster_method
old_block = '''        cluster = KMeans(
            num_cluster=args.intent_num,
            seed=1,
            hidden_size=64,
            gpu_id=args.gpu_id,
            device=torch.device("cuda"),
        )'''
new_block = '''        _cluster_method = getattr(args, "cluster_method", "kmeans")
        _ClusterClass = GMM if _cluster_method == "gmm" else KMeans
        cluster = _ClusterClass(
            num_cluster=args.intent_num,
            seed=1,
            hidden_size=64,
            gpu_id=args.gpu_id,
            device=torch.device("cuda"),
        )'''
if old_block in tcode:
    tcode = tcode.replace(old_block, new_block)
    print('Patched trainer to switch between KMeans and GMM')
elif '_ClusterClass' in tcode:
    print('Trainer already patched')
else:
    print('WARNING: could not find the cluster instantiation block. Inspect trainers.py manually.')

with open(trainers_path, 'w') as f:
    f.write(tcode)

Trainer already patched


## Step 4: Add the `--cluster_method` command-line argument

We register the new argument in `main.py` so it can be passed on the command line.

In [4]:
main_path = '/content/ICSRec/src/main.py'
with open(main_path) as f:
    mcode = f.read()

# Find a parser.add_argument line and insert our new argument right after the first one
if '--cluster_method' not in mcode:
    anchor = mcode.find('parser.add_argument(')
    insertion = ('parser.add_argument("--cluster_method", default="kmeans", type=str, '
                 'help="kmeans (original) or gmm (our improvement)")\n    ')
    mcode = mcode[:anchor] + insertion + mcode[anchor:]
    with open(main_path, 'w') as f:
        f.write(mcode)
    print('Added --cluster_method argument to main.py')
else:
    print('--cluster_method already present')

--cluster_method already present


## Step 5: Train ICSRec with GMM clustering

We compare GMM vs K-means at `intent_num=128` (see the markdown above for why we don't use the K=512 winner from our tuning sweep). Only `--cluster_method gmm` differs from the K-means K=128 run. We use `--model_idx 11` so the GMM run's files don't collide with the K-means runs (0-3).

**Note on speed:** GMM on CPU is slower than faiss K-means, so each epoch's clustering step will take longer. Expect a longer total run. On A100 the transformer training is fast, but the GMM fit happens on CPU each epoch.

In [5]:
%cd /content/ICSRec/src
!python main.py \
    --data_name Grocery_and_Gourmet_Food \
    --rec_weight 1.0 \
    --lambda_0 0.3 \
    --beta_0 0.1 \
    --f_neg \
    --intent_num 128 \
    --hidden_dropout_prob 0.5 \
    --attention_probs_dropout_prob 0.5 \
    --cluster_method gmm \
    --model_idx 11 \
    --epochs 200

/content/ICSRec/src
Using Cuda: True
--------------------Configure Info:------------
cluster_method                 :                                 gmm
data_dir                       :                            ../data/
output_dir                     :                              output
data_name                      :            Grocery_and_Gourmet_Food
encoder                        :                                 SAS
do_eval                        :                                   0
model_idx                      :                                  11
gpu_id                         :                                   0
noise_ratio                    :                                 0.0
temperature                    :                                 1.0
intent_num                     :                                 128
sim                            :                                 dot
model_name                     :                              ICSRec
hidden_size       

## Step 6: Back up the GMM run and read its result

In [6]:
import os, shutil
backup_dir = os.path.join(DRIVE_DIR, 'improvement_gmm')
os.makedirs(backup_dir, exist_ok=True)
for fn in os.listdir('/content/ICSRec/src/output'):
    if 'Grocery_and_Gourmet_Food-11' in fn:
        shutil.copy(f'/content/ICSRec/src/output/{fn}', f'{backup_dir}/{fn}')
        print(f'Backed up: {fn}')

log = '/content/ICSRec/src/output/ICSRec-SAS-Grocery_and_Gourmet_Food-11.txt'
if os.path.exists(log):
    print('\n===== Final result for GMM (intent_num=128) =====')
    with open(log) as f:
        for line in f.readlines()[-3:]:
            print(line.rstrip())

Backed up: ICSRec-SAS-Grocery_and_Gourmet_Food-11.pt
Backed up: ICSRec-SAS-Grocery_and_Gourmet_Food-11.txt

===== Final result for GMM (intent_num=128) =====
{'Epoch': 0, 'HIT@5': '0.0319', 'NDCG@5': '0.0209', 'HIT@10': '0.0523', 'NDCG@10': '0.0274', 'HIT@20': '0.0834', 'NDCG@20': '0.0352'}
ICSRec-SAS-Grocery_and_Gourmet_Food-11
{'Epoch': 0, 'HIT@5': '0.0319', 'NDCG@5': '0.0209', 'HIT@10': '0.0523', 'NDCG@10': '0.0274', 'HIT@20': '0.0834', 'NDCG@20': '0.0352'}


## Step 7: Compare GMM vs best K-means

This prints a clean comparison of the GMM improvement against the K-means baseline (intent_num=128).

In [7]:
import os, re

def read_test_metrics(log_path):
    if not os.path.exists(log_path):
        return None
    with open(log_path) as f:
        lines = f.readlines()[-10:]
    for line in reversed(lines):
        if 'HIT@5' in line:
            d = {}
            for key in ['HIT@5','NDCG@5','HIT@10','NDCG@10','HIT@20','NDCG@20']:
                m = re.search(key + r"'?:\s*'?([0-9.]+)", line)
                if m:
                    d[key] = m.group(1)
            return d
    return None

# K-means @ 128 = Run 2 (model_idx 1)
kmeans_log = '/content/ICSRec/src/output/ICSRec-SAS-Grocery_and_Gourmet_Food-1.txt'
if not os.path.exists(kmeans_log):
    drive_km = os.path.join(DRIVE_DIR, 'tuning_intent128', 'ICSRec-SAS-Grocery_and_Gourmet_Food-1.txt')
    if os.path.exists(drive_km):
        kmeans_log = drive_km

# GMM @ 128 = model_idx 11
gmm_log = '/content/ICSRec/src/output/ICSRec-SAS-Grocery_and_Gourmet_Food-11.txt'

km = read_test_metrics(kmeans_log)
gm = read_test_metrics(gmm_log)

keys = ['HIT@5','NDCG@5','HIT@10','NDCG@10','HIT@20','NDCG@20']
print('=' * 70)
print('Methodology improvement: GMM vs K-means (both intent_num=128)')
print('=' * 70)
print(f'{"method":<14}' + ''.join(f'{k:<10}' for k in keys))
print('-' * 70)
if km:
    print(f'{"K-means":<14}' + ''.join(f'{km.get(k,"-"):<10}' for k in keys))
if gm:
    print(f'{"GMM (ours)":<14}' + ''.join(f'{gm.get(k,"-"):<10}' for k in keys))

if km and gm:
    out = os.path.join(DRIVE_DIR, 'improvement_gmm_comparison.txt')
    with open(out, 'w') as f:
        f.write('GMM vs K-means (intent_num=128) - final TEST metrics\n')
        f.write(f'{"method":<14}' + ''.join(f'{k:<10}' for k in keys) + '\n')
        f.write(f'{"K-means":<14}' + ''.join(f'{km.get(k,"-"):<10}' for k in keys) + '\n')
        f.write(f'{"GMM (ours)":<14}' + ''.join(f'{gm.get(k,"-"):<10}' for k in keys) + '\n')
    print(f'\nComparison saved to {out}')

Methodology improvement: GMM vs K-means (both intent_num=128)
method        HIT@5     NDCG@5    HIT@10    NDCG@10   HIT@20    NDCG@20   
----------------------------------------------------------------------
K-means       0.0337    0.0215    0.0531    0.0278    0.0830    0.0352    
GMM (ours)    0.0319    0.0209    0.0523    0.0274    0.0834    0.0352    

Comparison saved to /content/drive/MyDrive/ICSRec_project/improvement_gmm_comparison.txt


## Done :)